# Cyndx AI Engineering Challenge: MMR Search Algorithm

At Cyndx we develop AI and algorithms that perform search, information retrieval, and other related AI services with our large database of private companies. "Search quality" is a hard-to-define metric that we think about a lot. One potential way to improve the perceived quality of search results is to reduce redundancy and increase diversity in the results returned to the user. Maximal Marginal Relevance (MMR) is a simple algorithm that accomplishes this goal.

## The Challenge

Implement a basic search algorithm that utilizes MMR to re-rank the search results.

### Requirements
1. Implement the MMR algorithm defined in the original paper: https://www.cs.cmu.edu/~jgc/publication/The_Use_MMR_Diversity_Based_LTMIR_1998.pdf
2. Utilize the provided dataset `./companies.csv` to demonstrate your implementation
3. The search should be composed of two parts:

    * A basic search algorithm with the signature `def search(query: str, ...) -> pd.DataFrame` that returns relevancy-based results for the given query
    * An MMR re-ranking algorithm with the signature `def rerank(results: pd.DataFrame, ...) -> pd.DataFrame`, returning the optimized results
    * The demonstration should compose these functions, for example: `rerank(search(query, ...), ...)`
4. After you have implemented and demonstrated your algorithms, please answer the questions at the bottom of this notebook
5. Please provide a `README.md` file with instructions on how to run your code, and a `requirements.txt` file with the necessary dependencies
6. Email us your solution as a .zip file containing your modified notebook and all necessary files

In [1]:
!pip3 install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [2]:
# Basic Python libraries
import os
import pandas as pd
import numpy as np
import seaborn as sns

# Used-defined libraries
from UDFs import clean_text
from UDFs import generate_companies_embeddings
from UDFs import mmr

# Sentence Transformers
from sentence_transformers import SentenceTransformer

# sklearn libraries
from sklearn.metrics.pairwise import cosine_similarity

/Users/kavisanthoshkumar/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### Implementation

In [3]:
companies = pd.read_csv('./companies.csv')
companies.head()

,Name,EmployeeCount,Description,Url,Region,Country,MetroArea,City
0,Molan Steel Co.,11-50,Molan Steel Co. engages in supplying steel pro...,molansteel.com,RY,SA,Riyadh/Saudi Arabia Metro,Riyadh
1,Hunter Douglas NV,10001+,"Hunter Douglas NV engages in the design, manuf...",hunterdouglas.com,ZH,NL,Amsterdam/NL Metro,Rotterdam
2,"Root, Inc.",501-1000,"Root, Inc. is a technology insurance company, ...",inc.joinroot.com,OH,US,Columbus/OH Metro,Columbus
3,Bowlero Corp.,10001+,Bowlero Corp. engages in operating bowling cen...,bowlero.com,VA,US,Richmond/VA Metro,Mechanicsville
4,Meridia Real Estate III SOCIMI SA,1-10,Meridia Real Estate III SOCIMI SA operates as ...,meridiarealestateiiisocimi.com,CT,ES,Barcelona/Spain Metro,Barcelona


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [5]:
def search(query:str, df: pd.DataFrame, n=100) -> pd.DataFrame:
    # Clean the query
    query = clean_text(query)
    
    # Sentence Transformers can be used for more advanced search capabilities
    model = SentenceTransformer("all-MiniLM-L6-v2")

    # Generate embeddings for the companies
    df = generate_companies_embeddings(df, model)
    
    # Generate embeddings for the query
    query_embedding = model.encode([query])

    # Compute the cosine similarity between the query and the company embeddings
    df["similarity"] = df["embeddings"].apply(lambda x: cosine_similarity(query_embedding, [x])[0][0])

    # Get the top n results based on similarity
    results = df.sort_values(by = "similarity", ascending =False).reset_index(drop=True)
    results["query_embeddings"] = [query_embedding] * len(results)

    ### TODO: replace this with your own search algo:
    results = results.loc[:n, :].copy()

    return results

In [ ]:
results = search(query = "companies manufacturing steel", 
                     df = companies)

In [ ]:
def rerank(results:pd.DataFrame, n=10) -> pd.DataFrame:

    # Query embeddings
    query_embeddings = results["query_embeddings"][0][0]
    
    # Extract document embeddings as a numpy array
    doc_embeddings = np.vstack(results["embeddings"].values)

    # Compute selected indices using MMR
    selected_indices = mmr(doc_embeddings, query_embeddings, n, 0.5)

    # Compute the MMR(Maximum Marginal Relevance) to rerank the results
    final_match = results.iloc[selected_indices, :].copy()
    
    return final_match


rerank(results=results, n=10)

,Name,EmployeeCount,Description,Url,Region,Country,MetroArea,City,concatenated_name_description,concatenated_cleaned_text,embeddings,similarity,query_embeddings
0,United States Steel Corp.,10001+,United States Steel Corp. engages in the manuf...,ussteel.com,PA,US,Pittsburgh/PA Metro,Pittsburgh,United States Steel Corp. United States Steel ...,united states steel corp united states steel c...,"[-0.03792182728648186, 0.011723068542778492, -...",0.701850,"[[-0.09800454, -0.009112361, -0.0067489683, 0...."
71,Boryszew SA,5001-10000,Boryszew SA engages in the production and sale...,boryszew.com.pl,MZ,PL,Warsaw/Poland Metro,Warsaw,Boryszew SA Boryszew SA engages in the product...,boryszew sa boryszew sa engages in the product...,"[-0.022545911371707916, -0.014332737773656845,...",0.494258,"[[-0.09800454, -0.009112361, -0.0067489683, 0...."
4,HG Metal Manufacturing Ltd.,101-250,HG Metal Manufacturing Ltd. is an investment h...,hgmetal.com,SW,SG,Singapore Metro,Singapore,HG Metal Manufacturing Ltd. HG Metal Manufactu...,hg metal manufacturing ltd hg metal manufactur...,"[-0.04993709176778793, 0.02958608604967594, 0....",0.612718,"[[-0.09800454, -0.009112361, -0.0067489683, 0...."
17,Insimbi Industrial Holdings Ltd.,501-1000,Insimbi Industrial Holdings Ltd. engages in th...,insimbi-group.co.za,GT,ZA,Johannesburg/S.Afr. Metro,Germiston,Insimbi Industrial Holdings Ltd. Insimbi Indus...,insimbi industrial holdings ltd insimbi indust...,"[-0.1477467119693756, 0.03411097079515457, -0....",0.566412,"[[-0.09800454, -0.009112361, -0.0067489683, 0...."
37,Termovent SC Livnica Celika AD,101-250,Termovent SC Livnica Celika AD engages in the ...,livnica.com,NB,RS,Belgrade/Serbia Metro,Backa Topola,Termovent SC Livnica Celika AD Termovent SC Li...,termovent sc livnica celika ad termovent sc li...,"[0.006184990517795086, -0.04094170778989792, -...",0.535387,"[[-0.09800454, -0.009112361, -0.0067489683, 0...."
6,Molan Steel Co.,11-50,Molan Steel Co. engages in supplying steel pro...,molansteel.com,RY,SA,Riyadh/Saudi Arabia Metro,Riyadh,Molan Steel Co. Molan Steel Co. engages in sup...,molan steel co molan steel co engages in suppl...,"[-0.1040932834148407, -0.011424730531871319, -...",0.609475,"[[-0.09800454, -0.009112361, -0.0067489683, 0...."
14,Argent Industrial Ltd.,1001-5000,Argent Industrial Ltd. operates as a holding c...,argent.co.za,NL,ZA,Johannesburg/S.Afr. Metro,Durban,Argent Industrial Ltd. Argent Industrial Ltd. ...,argent industrial ltd argent industrial ltd op...,"[-0.07139120250940323, -0.06429070979356766, -...",0.581966,"[[-0.09800454, -0.009112361, -0.0067489683, 0...."
25,Energomashspetsstal JSC,1001-5000,Energomashspetsstal JSC engages in the manufac...,emss.ua,DO,UA,Kyiv/Ukraine Metro,Kramatorsk,Energomashspetsstal JSC Energomashspetsstal JS...,energomashspetsstal jsc energomashspetsstal js...,"[-0.08374488353729248, 0.045263998210430145, -...",0.550038,"[[-0.09800454, -0.009112361, -0.0067489683, 0...."
12,S.S. Steel Ltd.,501-1000,S.S. Steel Ltd. engages in the business of man...,sssteel.biz,DA,BD,Asia (South/West) Metro,North Badda,S.S. Steel Ltd. S.S. Steel Ltd. engages in the...,ss steel ltd ss steel ltd engages in the busin...,"[-0.09782950580120087, -0.0389372780919075, -0...",0.588473,"[[-0.09800454, -0.009112361, -0.0067489683, 0...."
57,Sanoyas Holdings Corp.,501-1000,Sanoyas Holdings Corp. engages in the manageme...,sanoyas.co.jp,OS,JP,Osaka/Japan Metro,Osaka,Sanoyas Holdings Corp. Sanoyas Holdings Corp. ...,sanoyas holdings corp sanoyas holdings corp en...,"[-0.008469284512102604, -0.0183517187833786, 0...",0.507096,"[[-0.09800454, -0.009112361, -0.0067489683, 0...."


### Demo

In [ ]:
rerank(search('companies who manufacture steel products', companies))

### Questions

#### 1. Breifly summarize your implementation

#### 2. What would be your next step(s) to increase performance?

#### 3. What would be your next step(s) to improve search quality?